In [2]:
%pip install opencv-python
%pip install mediapipe --no-deps

Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.


In [3]:
import cv2

In [4]:
import mediapipe as mp

In [5]:
import math 

In [6]:
%pip install pycaw

Note: you may need to restart the kernel to use updated packages.


In [7]:
from ctypes import cast, POINTER
from comtypes import CLSCTX_ALL
from pycaw.pycaw import AudioUtilities, IAudioEndpointVolume

In [8]:
import numpy as np

In [9]:
devices = AudioUtilities.GetSpeakers()
if hasattr(devices, "Activate"):
    interface = devices.Activate(
        IAudioEndpointVolume._iid_, CLSCTX_ALL, None)
    volume = cast(interface, POINTER(IAudioEndpointVolume))
else:
    volume = devices.EndpointVolume

In [10]:
#volume.GetMute()
#print(volume.GetMasterVolumeLevel())
#print(volume.GetVolumeRange())


In [8]:
import os
import time
import urllib.request
import ctypes
import cv2
import mediapipe as mp
import numpy as np
import math

from ctypes import cast, POINTER
from comtypes import CLSCTX_ALL
from pycaw.pycaw import AudioUtilities, IAudioEndpointVolume
from mediapipe.tasks import python
from mediapipe.tasks.python import vision

devices = AudioUtilities.GetSpeakers()
if hasattr(devices, "Activate"):
    interface = devices.Activate(
        IAudioEndpointVolume._iid_, CLSCTX_ALL, None)
    volume = cast(interface, POINTER(IAudioEndpointVolume))
else:
    volume = devices.EndpointVolume

cap = cv2.VideoCapture(0)
cap.set(cv2.CAP_PROP_FRAME_WIDTH, 640)
cap.set(cv2.CAP_PROP_FRAME_HEIGHT, 480)

model_path = "hand_landmarker.task"
if not os.path.exists(model_path):
    urllib.request.urlretrieve(
        "https://storage.googleapis.com/mediapipe-models/hand_landmarker/hand_landmarker/float16/1/hand_landmarker.task",
        model_path,
    )

base_options = python.BaseOptions(model_asset_path=model_path)
options = vision.HandLandmarkerOptions(
    base_options=base_options,
    running_mode=vision.RunningMode.VIDEO,
    num_hands=1,
    min_hand_detection_confidence=0.5,
    min_hand_presence_confidence=0.5,
    min_tracking_confidence=0.5,
    )
landmarker = vision.HandLandmarker.create_from_options(options)

user32 = ctypes.windll.user32
VK_VOLUME_UP = 0xAF
KEYEVENTF_KEYUP = 0x0002

def trigger_system_osd():
    user32.keybd_event(VK_VOLUME_UP, 0, 0, 0)
    user32.keybd_event(VK_VOLUME_UP, 0, KEYEVENTF_KEYUP, 0)

gesture_min = 35
gesture_max = 220
smoothing = 0.2
prev_scalar = volume.GetMasterVolumeLevelScalar()
last_timestamp_ms = 0
last_osd_trigger_time = 0.0

try:
    while True:
        success, img = cap.read()
        if not success:
            continue

        img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        mp_image = mp.Image(image_format=mp.ImageFormat.SRGB, data=img_rgb)
        timestamp_ms = int(time.time() * 1000)
        if timestamp_ms <= last_timestamp_ms:
            timestamp_ms = last_timestamp_ms + 1
        last_timestamp_ms = timestamp_ms

        try:
            results = landmarker.detect_for_video(mp_image, timestamp_ms)
        except KeyboardInterrupt:
            break

        if results.hand_landmarks:
            hand_lms = results.hand_landmarks[0]
            lmList = []
            h, w, _ = img.shape

            for idx, lm in enumerate(hand_lms):
                cx, cy = int(lm.x * w), int(lm.y * h)
                lmList.append([idx, cx, cy])

            x1, y1 = lmList[4][1], lmList[4][2]
            x2, y2 = lmList[8][1], lmList[8][2]

            cv2.circle(img, (x1, y1), 10, (255, 0, 9), cv2.FILLED)
            cv2.circle(img, (x2, y2), 10, (255, 0, 9), cv2.FILLED)
            cv2.line(img, (x1, y1), (x2, y2), (134, 1, 175), 3)

            length = math.hypot(x2 - x1, y2 - y1)
            clipped_length = float(np.clip(length, gesture_min, gesture_max))
            target_scalar = float(np.interp(clipped_length, [gesture_min, gesture_max], [0.0, 1.0]))

            if clipped_length >= gesture_max - 2:
                target_scalar = 1.0
            elif clipped_length <= gesture_min + 2:
                target_scalar = 0.0

            smooth_scalar = prev_scalar + smoothing * (target_scalar - prev_scalar)

            if abs(smooth_scalar - prev_scalar) > 0.003:
                volume.SetMasterVolumeLevelScalar(smooth_scalar, None)
                prev_scalar = smooth_scalar
            elif target_scalar in (0.0, 1.0) and abs(prev_scalar - target_scalar) > 0:
                volume.SetMasterVolumeLevelScalar(target_scalar, None)
                prev_scalar = target_scalar

            now = time.time()
            if now - last_osd_trigger_time > 0.35 and abs(target_scalar - prev_scalar) < 0.02:
                trigger_system_osd()
                volume.SetMasterVolumeLevelScalar(prev_scalar, None)
                last_osd_trigger_time = now

            if length < 50:
                z1 = (x1 + x2) // 2
                z2 = (y1 + y2) // 2
                cv2.circle(img, (z1, z2), 10, (255, 0, 9), cv2.FILLED)

        actual_scalar = float(volume.GetMasterVolumeLevelScalar())
        volBar = np.interp(actual_scalar, [0.0, 1.0], [400, 150])
        volPer = int(round(actual_scalar * 100))
        if actual_scalar >= 0.995:
            volPer = 100

        cv2.rectangle(img, (50, 150), (85, 400), (255, 0, 0), 3)
        cv2.rectangle(img, (50, int(volBar)), (85, 400), (255, 0, 0), cv2.FILLED)
        cv2.putText(img, f"{volPer}%", (25, 100), cv2.FONT_HERSHEY_DUPLEX, 1.5, (126, 58, 234), 2)
        cv2.putText(img, f"Range: {gesture_min}-{gesture_max}", (110, 40), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 255, 255), 2)

        cv2.imshow("Image", img)
        key = cv2.waitKey(1) & 0xFF
        if key == ord("q") or key == 27:
            break
finally:
    cap.release()
    cv2.destroyAllWindows()
    landmarker.close()

KeyboardInterrupt: 

In [ ]:
#length 50-300
#volRange -24 to +21